# Lab 2D: Vector Search in Python

**Time**: ~15 min  
**Environment**: Jupyter kernel in VS Code  
**Note**: Requires Cosmos DB account with vector capability enabled and Azure OpenAI resource

In this exercise you will explore semantic similarity search using Azure Cosmos DB vector capability.

## Prerequisites

- Python 3.10+ with `azure-cosmos`, `azure-identity`, `openai`, `numpy`, and `python-dotenv` installed: `pip install azure-cosmos azure-identity openai numpy python-dotenv`
- `COSMOS_ENDPOINT` environment variable set to your Cosmos DB account endpoint
- `EMBEDDINGS_ENDPOINT` environment variable set to your Foundry endpoint for embeddings
- `EMBEDDINGS_KEY` environment variable set to the API key for your Foundry endpoint embeddings
- `EMBEDDINGS_MODEL` environment variable set to the name of the Foundry model for embeddings

Run each cell in order to complete the steps.

## Step 0: Initialize Connection

Set up the Cosmos client connection and Azure OpenAI embeddings client.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
EMBEDDINGS_ENDPOINT = os.environ.get("EMBEDDINGS_ENDPOINT")
EMBEDDINGS_KEY = os.environ.get("EMBEDDINGS_KEY")
EMBEDDINGS_MODEL = os.environ.get("EMBEDDINGS_MODEL", "textembedding3small")
DB_NAME = "WorkshopData"
CONT_NAME = "Docs"

for var in ["COSMOS_ENDPOINT", "EMBEDDINGS_ENDPOINT", "EMBEDDINGS_KEY"]:
    if not os.environ.get(var):
        raise RuntimeError(f"{var} environment variable is required.")

print(f"Cosmos Endpoint:     {ENDPOINT}")
print(f"Embeddings Endpoint: {EMBEDDINGS_ENDPOINT}")
print(f"Database:            {DB_NAME}")
print(f"Container:           {CONT_NAME}")
print(f"Embeddings Model:    {EMBEDDINGS_MODEL}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential
from openai import OpenAI

# Cosmos DB client (Entra ID auth)
cred = DefaultAzureCredential()
cosmos_client = CosmosClient(url=ENDPOINT, credential=cred)
db = cosmos_client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to Cosmos DB: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

# Embeddings client: separate Azure OpenAI resource, API key auth
# (the v1 embeddings surface does not yet support Entra ID).
embeddings_client = OpenAI(
    base_url=f"{EMBEDDINGS_ENDPOINT.rstrip('/')}/openai/v1/",
    api_key=EMBEDDINGS_KEY,
)
print(f"Embeddings client initialized (deployment: {EMBEDDINGS_MODEL})")

## Step 1: Generate Embeddings (Prebuilt)

Creates 3 sample documents, generates embeddings using Azure OpenAI, and stores them in `WorkshopData/Docs`.

In [ ]:
def embed_text(text: str) -> list[float]:
    resp = embeddings_client.embeddings.create(input=text, model=EMBEDDINGS_MODEL)
    return resp.data[0].embedding


docs = [
    {"id": "d1", "title": "Global Distribution", "text": "Azure Cosmos DB replicates your data across regions worldwide for low-latency access.", "partitionKey": "docs"},
    {"id": "d2", "title": "Vector Search", "text": "Vector search retrieves documents by meaning, even when they share no keywords with the query.", "partitionKey": "docs"},
    {"id": "d3", "title": "Provisioned Throughput", "text": "Provisioned throughput reserves request units per second for predictable performance.", "partitionKey": "docs"},
]

for doc in docs:
    text = doc["text"]
    doc["embedding"] = embed_text(text)
    try:
        container.upsert_item(body=doc)
        ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])
        print(f"  Indexed: {doc['title']}")
        print(f"  RU charged: {ru}")
    except Exception as ex:
        print(f"  Error indexing: {ex}")

print(f"\nEmbedded {len(docs)} documents")

## Step 2: Vector Search

Replace the placeholder `vector_query` in the code cell with a `VectorDistance` query that returns the top 2 most-similar docs:

```python
vector_query = """
SELECT TOP 2 c.id, c.title, c.text, VectorDistance(c.embedding, @emb) AS score
FROM c
WHERE c.partitionKey = 'docs'
ORDER BY VectorDistance(c.embedding, @emb)
"""
```

**Expected output**: the top 2 documents most similar to the query. The **Vector Search** doc ranks first — matched by **meaning**, even though the query shares almost no keywords with it.

In [ ]:
search_text = "finding related information based on concepts instead of exact words"
print(f"Searching for: {search_text}")
print()

query_embedding = embed_text(search_text)

vector_query = """
SELECT TOP 2 c.id, c.title, c.text, VectorDistance(c.embedding, @emb) AS score
FROM c
WHERE c.partitionKey = 'docs'
ORDER BY VectorDistance(c.embedding, @emb)
"""

results = list(container.query_items(
    query=vector_query,
    parameters=[{"name": "@emb", "value": query_embedding}],
    enable_cross_partition_query=True
))

print("Vector search results:")
for r in results:
    print(f"  Title: {r.get('title')}")
    print(f"  Text:  {r.get('text')}")
    print(f"  Score: {r.get('score')}")
    print()

## Step 3: Full-Text Search

Replace the placeholder `fts_query` in the code cell with a `FullTextContains` query:

```python
fts_query = """
SELECT * FROM c WHERE FullTextContains(c.text, @search) AND c.partitionKey = 'docs'
"""
```

**Expected output**: the **Provisioned Throughput** doc — the only one whose `text` literally contains the word `throughput`. Full-text matches exact keywords; Step 2 matched by meaning.

In [ ]:
search_text = "throughput"
print(f"Searching for: {search_text}")
print()

fts_query = """
SELECT * FROM c WHERE FullTextContains(c.text, @search) AND c.partitionKey = 'docs'
"""

results = list(container.query_items(
    query=fts_query,
    parameters=[{"name": "@search", "value": search_text}],
    enable_cross_partition_query=True
))

print(f"Full-text search results for '{search_text}':")
for r in results:
    print(f"  ID:    {r.get('id')}")
    print(f"  Title: {r.get('title')}")
    print(f"  Text:  {r.get('text')}")
    print()